# 7장 · Python 디버깅과 Git 변경 이력

디버깅은 예상값과 실행값이 처음 달라진 줄을 찾는 작업입니다. 수정한 뒤에는 Git 변경 내역에서 그 줄을 다시 확인할 수 있습니다.

학습 기록은 30분, 20분, 40분인데 프로그램은 합계로 40분을 냅니다. 같은 입력으로 증상을 다시 만들고, 고친 줄을 동료에게 어떻게 보여 줄지 실습합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## for 반복문과 누적 변수의 초기화 위치

디버깅은 막연히 코드를 바꾸기 전에 틀린 결과를 같은 입력으로 다시 만드는 일부터 시작합니다. 반복을 한 바퀴 돌 때마다 total에 직전 바퀴까지 더한 값이 남아 있어야 합니다.

<!-- lesson-visual practice-space -->
**초기화 위치를 옮기면 마지막 합은?**

세 기록의 시간은 30분, 20분, 40분입니다. 같은 자료를 두 번 계산하고, 두 번째 기록을 읽은 직후의 막대부터 비교하세요.

반복문 안에서 초기화하면 30 → 20 → 40분, 반복 전에 한 번만 초기화하면 30 → 50 → 90분입니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter07/#practice-space)
<!-- /lesson-visual practice-space -->


### 셀 01 · 누적 변수 재초기화 오류

첫 번째 반복이 끝났을 때의 total을 떠올리고 둘째 반복의 첫 줄을 읽으세요.


In [1]:
records = [{"minutes": 30}, {"minutes": 20}, {"minutes": 40}]
total = 0
for record in records:
    total = 0
    total += record["minutes"]
print(total)

40


40이 나옵니다. 30과 20이 더해지기는 하지만 다음 바퀴마다 다시 0이 됩니다. 마지막 바퀴에서 더한 40만 남습니다.


### 누적 변수의 마지막 값을 추적하기

<p>30, 20, 40을 더했는데 40만 남았습니다. total = 0이 언제 실행되는지 세 바퀴에 각각 표시해 보세요. 초기화를 반복문 바깥으로 옮기면 두 번째 바퀴는 첫 번째 바퀴가 남긴 30에서 시작합니다.</p>


### 과제 02 · total_minutes() 반환값 검증

total 초기화 위치를 바꿔 세 건의 합을 돌려주는 함수를 작성하세요. 입력 자료는 그대로 씁니다.


In [2]:
def total_minutes(records):
    pass  # 세 기록의 분을 더해 반환하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def total_minutes(records):
    total = 0
    for record in records:
        total += record[&quot;minutes&quot;]
    return total

assert total_minutes(records) == 90
print(total_minutes(records))</code></pre>
<p>실행 결과</p><pre><code>90</code></pre>
<p>90이면 세 기록의 합이 맞습니다. 40으로 되돌아가면 assert에서 멈추므로 고친 문제의 재발을 바로 찾을 수 있습니다.</p>
</details>


### 테스트에서 확인할 두 합계

<p>total_minutes에 세 기록을 넣으면 기대 합계는 90입니다. 빈 목록을 넣었을 때는 0이어야 합니다. 노트북에서 두 결과를 각각 확인하고, 이후 pytest로 옮길 때도 두 입력의 기대값을 따로 적으세요.</p>


## unified_diff()로 변경 줄 비교

Git의 diff에서 -로 시작하는 본문 줄은 이전 내용, +는 새 내용입니다. 아래에서는 같은 형식의 차이를 표준 라이브러리로 만들어 읽습니다. 앞뒤 문맥이 없으면 초기화한 위치를 오해하기 쉽습니다.


### 셀 03 · unified_diff()의 삭제 줄

잘못된 계산에서 반복문 안쪽 초기화 한 줄을 뺐습니다.


In [3]:
from difflib import unified_diff
before = ["total = 0", "for record in records:", "    total = 0", '    total += record["minutes"]']
after = [before[0], before[1], before[3]]
print("\n".join(unified_diff(before, after, fromfile="이전", tofile="수정", lineterm="")))

--- 이전
+++ 수정
@@ -1,4 +1,3 @@
 total = 0
 for record in records:
-    total = 0
     total += record["minutes"]


마이너스가 붙은 total = 0 줄 앞에 들여쓰기가 있습니다. 바깥의 초기화는 남아 있어 첫 바퀴도 정상적으로 시작합니다.


### 변경 diff 검토

<p>리뷰에서는 변경 목적, 영향을 받는 입력, 실행한 검사, 아직 확인하지 못한 환경을 전달합니다. “잘 됩니다” 대신 어떤 명령과 입력에서 어떤 결과를 얻었는지 적습니다.</p>

<p>실제 원격 협업에서는 담당 저장소·브랜치와 업로드 권한을 먼저 확인합니다. fetch는 원격 이력을 가져오고, pull은 가져온 이력을 현재 브랜치에 통합하며, push는 로컬 이력을 원격에 보냅니다. 이번 8H 기본 실습은 로컬 이력과 충돌 해결을 대상으로 하며 원격 업로드 성공을 검증하지 않습니다.</p>

<table><caption>짝 리뷰 체크</caption><thead><tr><th scope="col">질문</th><th scope="col">확인 자료</th></tr>
</thead><tbody><tr><td>목적이 분명한가?</td><td>변경 설명 1~2문장</td></tr>
<tr><td>필요한 파일만 포함했는가?</td><td>git diff --cached·.gitignore</td></tr>
<tr><td>정상·실패를 확인했는가?</td><td>명령·입력·예상/실제 결과</td></tr>
<tr><td>동료가 실행할 수 있는가?</td><td>README·requirements</td></tr>
<tr><td>민감 값이 있는가?</td><td>.env·인증 헤더·로그 제외</td></tr>
</tbody></table>


## Git 병합 충돌 표시와 해결

서로 다른 브랜치에서 같은 줄을 다르게 고치면 Git이 자동으로 한쪽을 선택할 수 없습니다. 충돌 표시 위쪽과 아래쪽을 읽고 두 사람이 고친 의도를 살려 최종 문장을 씁니다.


### 두 브랜치가 같은 줄을 바꿨을 때

<p>브랜치가 갈라진 뒤 서로 같은 줄을 다르게 고치면 Git이 어떤 문장을 남길지 자동으로 정하지 못할 수 있습니다. 이 셀에는 그때 볼 수 있는 충돌 내용을 문자열로 담았습니다.</p>


### 셀 04 · Git 충돌 마커 해석

수업용 안내문의 같은 줄에 서로 다른 실행 방법을 적었습니다.

아래 문자열에는 이미 충돌 표시가 들어 있습니다. 이번 셀에서는 양쪽이 바꾼 줄을 찾아보세요. 실제 저장소에서 병합을 마칠 때에는 별도로 파일의 충돌 표시를 정리하고 선택한 내용을 기록합니다.


In [4]:
conflict = """<<<<<<< HEAD
전체 셀을 실행하세요.
=======
새 커널에서 시작하세요.
>>>>>>> partner"""
print(conflict)

<<<<<<< HEAD
전체 셀을 실행하세요.
새 커널에서 시작하세요.
>>>> partner


HEAD 쪽은 전체 실행을, partner 쪽은 새 커널로 시작할 것을 강조합니다. 구분선 자체는 안내문 내용에 넣지 않습니다. 실제 브랜치에서 해결한 파일은 다시 스테이징하고 병합을 마칩니다.


### 동일한 줄의 병렬 변경

<p>git diff는 아직 스테이징하지 않은 변경을 보여 줍니다. 줄 앞의 -는 이전 내용, +는 새 내용입니다. 파일 변경을 확인한 뒤 git add로 선택하고 커밋합니다.</p>


### 과제 05 · 충돌 해결 후 재현 절차

두 안내를 한 문장으로 합치고 30·20·40분의 기대 합계까지 적으세요.


In [5]:
# 충돌 표시 없이 인수인계 문장을 완성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(&quot;새 커널에서 전체 셀을 실행하세요. 30·20·40분의 합은 90분입니다.&quot;)</code></pre>
<p>실행 결과</p><pre><code>새 커널에서 전체 셀을 실행하세요. 30·20·40분의 합은 90분입니다.</code></pre>
<p>동료가 같은 입력과 상태에서 다시 확인할 수 있습니다. pytest로 검사를 묶을 때도 위의 assert처럼 입력과 기대값을 가까이 둡니다.</p>
</details>


### 충돌 마커 안의 두 지시 합치기

<p>HEAD 쪽 문장은 ‘전체 셀을 실행하세요.’이고 partner 쪽 문장은 ‘새 커널에서 시작하세요.’입니다. 실행 순서를 생각해 새 커널을 먼저 시작하고 전체 셀을 실행하라는 한 문장으로 합치세요. 결과 문장에는 충돌 마커를 넣지 않습니다.</p>


## 돌아보기

기록을 한 건만 넣으면 초기화가 잘못된 코드도 우연히 통과합니다. 빈 목록과 서로 다른 분을 가진 두 건으로 새 검사를 만들어 보고, 어떤 입력에서 고치기 전·후가 구별되는지 설명해 보세요.
